# Data Ingestion and basic cleaning

Propósito: Cargar los datos crudos y guardarlos en un formato limpio.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', None)
pd.options.display.max_info_columns = 200

In [2]:
data_raw_accepted = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", low_memory=False)

In [3]:
diccionary = pd.read_csv("../data/raw/Lending_Club_Diccionario_Profesional.csv")
dictionary = dict(zip(diccionary['Variable'].str.strip(), diccionary['Descripcion_ES']))

In [4]:
data_raw_accepted.shape

(2260701, 151)

In [5]:
dictionary["loan_status"]

'El estado final o actual del préstamo (Pagado, Impago, Al día, etc.).'

## Paso 1: TARGET 

Esta es la decisión más importante para el proyecto. El objetivo de un modelo de riesgo es predecir si un préstamo será "Bueno" o "Malo".

Solo podemos entrenar el modelo usando préstamos que ya han terminado. No podemos usar préstamos que están "En Curso" (Current) porque aún no sabes cuál será su resultado final.


### 1.1. Filtrar y Eliminar (Los "Indecisos")
Estas son las filas que vamos a ELIMINAR del dataset de entrenamiento y prueba, porque su resultado final es desconocido.

In [6]:
data_raw_accepted["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

### Significado de los valores en `loan_status`

| Estado del préstamo                                              | Significado                                                                 |
|------------------------------------------------------------------|------------------------------------------------------------------------------|
| **Fully Paid** (1,076,751)                                       | El prestatario pagó el préstamo completo, incluyendo intereses y cargos.    |
| **Current** (878,317)                                            | El préstamo está activo y los pagos están al día.                            |
| **Charged Off** (268,559)                                        | El préstamo fue considerado incobrable y se dio de baja como pérdida.       |
| **Late (31-120 days)** (21,467)                                  | El prestatario tiene entre 31 y 120 días de atraso en el pago.              |
| **In Grace Period** (8,436)                                      | El prestatario no ha pagado a tiempo, pero aún tiene margen para hacerlo sin penalización.     |
| **Late (16-30 days)** (4,349)                                    | El prestatario tiene entre 16 y 30 días de atraso en el pago.               |
| **Does not meet the credit policy. Status: Fully Paid** (1,988)  | El préstamo fue pagado, pero no cumplía con ciertos criterios de política.  |
| **Does not meet the credit policy. Status: Charged Off** (761)   | El préstamo fue incobrable y tampoco cumplía con la política de crédito.    |
| **Default** (40)                                                 | El prestatario incumplió el préstamo, pero aún no se ha dado de baja total. |

In [7]:
# 1. Definir las listas de estados
bad_status = ["Charged Off", "Late (31-120 days)", "Does not meet the credit policy. Status:Charged Off", "Default"]
indecisos = ["Current", "In Grace Period", "Late (16-30 days)"]
good_status = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]


# 2. Filtrar el DataFrame para quedarnos SOLO con los préstamos terminados
df_completed_loans = data_raw_accepted[data_raw_accepted['loan_status'].isin(good_status + bad_status)].copy()

# 3. Crear la variable objetivo binaria (0 o 1)
df_completed_loans["loan_status"] = df_completed_loans['loan_status'].apply(
    lambda x: 1 if x in bad_status else 0
)

df_completed_loans["loan_status"].value_counts()

loan_status
0    1078739
1     290827
Name: count, dtype: int64

## PASO 2: Data Leakage
Eliminar variables de  (Fuga de datos)

Estas son las variables "trampa". Son datos que solo puedes saber después de que el préstamo ha sido aprobado y el cliente ha empezado a pagar. Si las incluyes, tu modelo parecerá perfecto en el entrenamiento, pero fallará en el mundo real.

"¿Podría saber esta información en el momento exacto en que el cliente solicita el préstamo?"

Si la respuesta es "No", es una fuga de datos y debe ser eliminada.

### 2.1. Variables de Fuga de Pagos

Estas variables describen el comportamiento de pago del préstamo. Solo pueden existir después de que el préstamo fue aprobado. 

In [8]:
# Estas variables describen el rendimiento del préstamo
fuga_pagos = [
    'collection_recovery_fee',
    'last_pymnt_amnt',
    'last_pymnt_d',
    'next_pymnt_d',
    'out_prncp',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_int',
    'total_rec_late_fee',
    'total_rec_prncp',
    'recoveries'
]

In [9]:
for var in fuga_pagos:
    print(var, " -> ", dictionary[var])

collection_recovery_fee  ->  Tarifa cobrada por la recuperación de un préstamo después de un impago.
last_pymnt_amnt  ->  Monto del último pago recibido de este préstamo.
last_pymnt_d  ->  Fecha (mes/año) del último pago recibido.
next_pymnt_d  ->  Fecha programada para el próximo pago de este préstamo.
out_prncp  ->  Cuánto capital (principal) del préstamo le falta por pagar.
out_prncp_inv  ->  Cuánto capital le falta por pagar a los inversores.
total_pymnt  ->  Monto total que el cliente ha pagado de este préstamo hasta la fecha.
total_pymnt_inv  ->  Monto total que los inversores han recibido de este préstamo.
total_rec_int  ->  Monto total de interés que ha pagado este préstamo.
total_rec_late_fee  ->  Monto total de multas por atraso que ha pagado este préstamo.
total_rec_prncp  ->  Monto total de capital (principal) que ha pagado este préstamo.
recoveries  ->  Dinero recuperado por la agencia de cobranzas *después* de que el préstamo falló.


### 2.2. Variables de Fuga de Dificultades

Estas variables solo se llenan si un cliente tiene problemas para pagar y entra en un plan especial. Esto ocurre después de la aprobación.

In [10]:
# Estas variables solo existen si el prestatario tiene problemas
fuga_hardship = [
    'hardship_flag',
    'hardship_type',
    'hardship_reason',
    'hardship_status',
    'deferral_term',
    'hardship_amount',
    'hardship_start_date',
    'hardship_end_date',
    'payment_plan_start_date',
    'hardship_length',
    'hardship_dpd',
    'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount'
]

In [11]:
for var in fuga_hardship:
    print(var, " -> ", dictionary[var])

hardship_flag  ->  Si el cliente está en un plan de "dificultad de pago".
hardship_type  ->  El tipo de plan de dificultad.
hardship_reason  ->  La razón del plan de dificultad.
hardship_status  ->  El estado del plan de dificultad.
deferral_term  ->  Cuántos meses se aplazó el pago.
hardship_amount  ->  El monto del pago reducido durante el plan de dificultad.
hardship_start_date  ->  Fecha de inicio del plan de dificultad.
hardship_end_date  ->  Fecha de fin del plan de dificultad.
payment_plan_start_date  ->  Fecha de inicio del plan de pago de dificultad.
hardship_length  ->  Duración (en meses) del plan de dificultad.
hardship_dpd  ->  Días de atraso que tenía el cliente al iniciar el plan de dificultad.
hardship_loan_status  ->  Estado del préstamo al iniciar el plan de dificultad.
orig_projected_additional_accrued_interest  ->  Interés adicional proyectado debido al plan de dificultad.
hardship_payoff_balance_amount  ->  Saldo deudor al iniciar el plan de dificultad.
hardship_la

### 2.3. Variables de Fuga de Liquidación (Settlement)
Estas variables solo se llenan si el préstamo entra en impago (Charged Off) y se negocia un acuerdo. Son un sinónimo de tu variable objetivo.

In [12]:
# Estas variables solo existen si el préstamo ha fallado
fuga_settlement = [
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term'
]

In [13]:
for var in fuga_settlement:
    print(var, " -> ", dictionary[var])

debt_settlement_flag  ->  Si el cliente (que ya falló) está en un acuerdo de liquidación de deuda.
debt_settlement_flag_date  ->  Fecha del acuerdo de liquidación.
settlement_status  ->  Estado del acuerdo de liquidación.
settlement_date  ->  Fecha del acuerdo de liquidación.
settlement_amount  ->  Monto acordado para liquidar la deuda.
settlement_percentage  ->  Porcentaje del saldo que se acordó pagar.
settlement_term  ->  Plazo (en meses) del acuerdo de liquidación.


### 2.4. Variables que no aportan información

In [14]:
# IDs, URLs, o texto libre que no usaremos
columnas_inutiles = [
    'id',
    'member_id',
    'url',
    'emp_title', # Demasiados valores únicos (31 000), y contenido engañoso
    'desc',      # Texto libre - 90% nulos
    'title',     # Texto libre, redundante con 'purpose'
    'zip_code'   # Usar 'addr_state' en su lugar, solo son los 3 primeros digitos
]

In [15]:
for var in columnas_inutiles:
    print(var, " -> ", dictionary[var])

id  ->  ID único asignado por LendingClub al préstamo.
member_id  ->  ID único asignado por LendingClub al cliente.
url  ->  La dirección web de la página del préstamo.
emp_title  ->  El cargo laboral autoinformado por el cliente (datos de texto libre).
desc  ->  Una descripción en texto libre proporcionada por el cliente sobre el préstamo.
title  ->  El título en texto libre que el cliente escribió para su préstamo.
zip_code  ->  Los 3 primeros dígitos del código postal del cliente.


In [16]:
df_filtered_data_leakege = df_completed_loans.drop(columns=fuga_pagos + fuga_hardship + fuga_settlement + columnas_inutiles)

In [17]:
df_filtered_data_leakege.shape

(1369566, 110)

## PASO 3: Valores Nulos

Este es el primer filtro 

In [18]:
# calcular porcentaje de nulos por variable
vacios_features = df_filtered_data_leakege.isnull().sum() * 100 / len(df_filtered_data_leakege)

# convertir a DataFrame, añadir descripción y filtrar > 50%
vacios_df = pd.DataFrame(vacios_features, columns=['Porcentaje_Nulos'])
vacios_df['Descripcion'] = vacios_df.index.map(dictionary)
vacios_df = vacios_df[vacios_df['Porcentaje_Nulos'] > 50].sort_values('Porcentaje_Nulos', ascending=False)

# imprimir ordenado y formateado
print(f"{'Variable':36} {'Descripción':150} {'% Nulos':>8}")
print("-" * 180)
for var, row in vacios_df.iterrows():
    desc = (row['Descripcion'] if pd.notna(row['Descripcion']) else "")[:150]
    print(f"{var:36} {desc:150} {row['Porcentaje_Nulos']:8.2f}%")

Variable                             Descripción                                                                                                                                             % Nulos
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
sec_app_mths_since_last_major_derog  Meses desde la última calificación derogatoria grave del solicitante secundario.                                                                          99.45%
sec_app_revol_util                   Tasa de utilización de crédito revolvente del solicitante secundario.                                                                                     98.52%
revol_bal_joint                      Saldo total de crédito revolvente combinado de los coprestatarios.                                                                                        98.49%
sec_app_earliest_cr_line    

Visualizamos que varios de los datos faltantes, se deben a informacion de participantes secundarios.
Nuestro análisis será individual, por ellos se liminarán las variables.

In [19]:
cols_duplicates = [col for col in df_filtered_data_leakege.columns 
                if col.startswith('sec_app_') or col.endswith('_joint')]

vacios_features[cols_duplicates]

annual_inc_joint                       97.956725
dti_joint                              97.956944
verification_status_joint              97.973446
revol_bal_joint                        98.492442
sec_app_fico_range_low                 98.492369
sec_app_fico_range_high                98.492369
sec_app_earliest_cr_line               98.492369
sec_app_inq_last_6mths                 98.492369
sec_app_mort_acc                       98.492369
sec_app_open_acc                       98.492369
sec_app_revol_util                     98.519166
sec_app_open_act_il                    98.492369
sec_app_num_rev_accts                  98.492369
sec_app_chargeoff_within_12_mths       98.492369
sec_app_collections_12_mths_ex_med     98.492369
sec_app_mths_since_last_major_derog    99.454134
dtype: float64

In [20]:
df_filtered_secundary = df_filtered_data_leakege.drop(columns=cols_duplicates)
df_filtered_secundary.shape

(1369566, 94)

In [21]:
vacios_features = df_filtered_secundary.isnull().sum() * 100 / len(df_filtered_secundary)

vacios_df = pd.DataFrame(vacios_features, columns=['Porcentaje_Nulos'])
vacios_df['Descripcion'] = vacios_df.index.map(dictionary)

vacios_df = vacios_df[vacios_df['Porcentaje_Nulos'] > 50].sort_values('Porcentaje_Nulos', ascending=False)


print(f"{'Variable':32} {'Descripción':100} {'% Nulos':>8}")
print("-" * 180)
for var, row in vacios_df.iterrows():
    desc = (row['Descripcion'] if pd.notna(row['Descripcion']) else "")[:100]
    print(f"{var:32} {desc:100} {row['Porcentaje_Nulos']:8.2f}%")

Variable                         Descripción                                                                                           % Nulos
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
mths_since_last_record           Cuántos meses han pasado desde su último registro público negativo.                                     82.95%
mths_since_recent_bc_dlq         Cuántos meses han pasado desde que se atrasó en un pago de tarjeta por última vez.                      76.31%
mths_since_last_major_derog      Cuántos meses han pasado desde la última vez que tuvo un evento crediticio grave (ej. quiebra).         73.69%
mths_since_recent_revol_delinq   Cuántos meses han pasado desde el último atraso en una tarjeta de crédito.                              66.59%
il_util                          Ratio de uso en préstamos a plazos (ej. cuánto debe de su préstamo 

Vamos a analizar los mas importantes, los demás lo vamos a eliminar

In [22]:
mths_data = [
    'mths_since_last_record',
    'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog',
    'mths_since_recent_revol_delinq',
    'mths_since_rcnt_il',
    'mths_since_last_delinq']

In [23]:
def analizar_variable_con_nulos(df, columna_a_investigar_nombre, columna_de_control_nombre):
    # 1. Rellenamos temporalmente los NaN con un string para que 'crosstab' los agrupe
    columna_a_investigar = df[columna_a_investigar_nombre].fillna('NaN_significativo')
    columna_de_control = df[columna_de_control_nombre]
    return pd.crosstab(columna_a_investigar, columna_de_control)

In [24]:
analizar_variable_con_nulos(
    df_filtered_secundary,
    'mths_since_last_delinq', #'Cuántos meses han pasado desde la última vez que el cliente tuvo un pago atrasado.'
    'delinq_2yrs' # 'Número de veces que el cliente se ha atrasado más de 30 días en un pago en los últimos 2 años.'
)

delinq_2yrs,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,36.0,39.0
mths_since_last_delinq,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0.0,821,683,327,175,55,32,18,10,2,1,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,1,2670,1211,620,265,128,51,24,15,6,6,3,9,3,2,2,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2.0,0,3661,1572,690,288,125,62,26,18,8,5,2,4,2,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3.0,1,4625,1973,844,335,157,68,41,17,18,8,3,2,3,1,0,1,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0
4.0,2,5371,2347,957,402,219,108,54,27,24,9,6,2,3,3,4,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188.0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
192.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
202.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:

analizar_variable_con_nulos(
    df_filtered_secundary,
    'mths_since_last_record', #'Cuántos meses han pasado desde la última vez que el cliente tuvo un registro público negativo.'
    'pub_rec') # 'Número de registros públicos negativos que tiene el cliente.'

pub_rec,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,28.0,34.0,37.0,40.0,44.0,45.0,46.0,47.0,49.0,54.0,61.0,63.0,86.0
mths_since_last_record,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0.0,1275,7,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,0,62,25,8,5,8,6,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2.0,0,63,20,11,11,4,3,3,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3.0,0,124,38,27,14,3,6,7,0,1,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4.0,0,123,59,24,17,12,3,5,3,2,1,2,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
123.0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
124.0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [26]:
analizar_variable_con_nulos(
    df_filtered_secundary,
    'mths_since_last_major_derog', #'Cuántos meses han pasado desde la última vez que el cliente tuvo un evento crediticio grave.'
    'num_accts_ever_120_pd' #'Número de cuentas que el cliente ha tenido con un retraso de pago de 120 días o más.'
)

num_accts_ever_120_pd,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,29.0,30.0,31.0,32.0,33.0,34.0,35.0,38.0,39.0,51.0
mths_since_last_major_derog,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0.0,67,105,49,17,15,5,4,3,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,224,381,176,87,45,33,16,18,8,8,1,1,0,0,2,2,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2.0,228,427,217,85,59,28,37,15,14,7,3,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3.0,272,483,236,108,48,24,30,14,16,6,5,2,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0
4.0,320,703,267,141,81,40,29,18,12,12,4,1,2,1,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
197.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
202.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Al analizar 3 del grupo de mths_, inferimos que los datos NaN no son ruido, sino que significa NUNCA.

¿Cuantos meses ha pasado desde que paso tal cosa?  El NaN significa NUNCA pasó, es una pista de ser variables importantes para incluirlo al modelo.

In [27]:
eliminar = list(set(vacios_df.index.tolist()) - set(mths_data))
eliminar

['inq_last_12m',
 'open_acc_6m',
 'all_util',
 'total_bal_il',
 'open_il_24m',
 'open_il_12m',
 'max_bal_bc',
 'total_cu_tl',
 'open_rv_12m',
 'open_act_il',
 'il_util',
 'inq_fi',
 'open_rv_24m']

In [28]:
df_completed_filtred = df_filtered_secundary.drop(columns=eliminar)
df_completed_filtred.shape

(1369566, 81)

Poniendo las Fechas en formato Datetime

In [16]:
select_not_float = df_completed_filtred.select_dtypes(exclude=['float64', 'int64'])
select_not_float.head()

,term,grade,sub_grade,emp_length,home_ownership,verification_status,issue_d,pymnt_plan,purpose,addr_state,earliest_cr_line,initial_list_status,last_credit_pull_d,application_type,disbursement_method
0,36 months,C,C4,10+ years,MORTGAGE,Not Verified,Dec-2015,n,debt_consolidation,PA,Aug-2003,w,Mar-2019,Individual,Cash
1,36 months,C,C1,10+ years,MORTGAGE,Not Verified,Dec-2015,n,small_business,SD,Dec-1999,w,Mar-2019,Individual,Cash
2,60 months,B,B4,10+ years,MORTGAGE,Not Verified,Dec-2015,n,home_improvement,IL,Aug-2000,w,Mar-2019,Joint App,Cash
3,60 months,F,F1,3 years,MORTGAGE,Source Verified,Dec-2015,n,major_purchase,PA,Jun-1998,w,Mar-2018,Individual,Cash
4,36 months,C,C3,4 years,RENT,Source Verified,Dec-2015,n,debt_consolidation,GA,Oct-1987,w,May-2017,Individual,Cash


In [17]:
fechas = ["issue_d", "earliest_cr_line","last_credit_pull_d"]

for fecha in fechas:
    df_completed_filtred[fecha] = pd.to_datetime(df_completed_filtred[fecha], format='%b-%Y')

In [18]:
df_completed_filtred.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,mths_since_rcnt_il,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method
0,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,2015-12-01,0,n,debt_consolidation,PA,5.91,0.0,2003-08-01,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,2019-03-01,564.0,560.0,0.0,30.0,1.0,Individual,0.0,722.0,144904.0,21.0,9300.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,Cash
1,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,2015-12-01,0,n,small_business,SD,16.06,1.0,1999-12-01,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,2019-03-01,699.0,695.0,0.0,NaN,1.0,Individual,0.0,0.0,204396.0,19.0,111800.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,Cash
2,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,2015-12-01,0,n,home_improvement,IL,10.78,0.0,2000-08-01,695.0,699.0,0.0,NaN,NaN,6.0,0.0,7869.0,56.2,18.0,w,2019-03-01,704.0,700.0,0.0,NaN,1.0,Joint App,0.0,0.0,189699.0,19.0,14000.0,6.0,31617.0,2737.0,55.9,0.0,0.0,125.0,184.0,14.0,14.0,5.0,101.0,NaN,10.0,NaN,0.0,2.0,3.0,2.0,4.0,6.0,4.0,7.0,3.0,6.0,0.0,0.0,0.0,0.0,100.0,50.0,0.0,0.0,218418.0,18696.0,6200.0,14877.0,Cash
3,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,2015-12-01,0,n,major_purchase,PA,25.37,1.0,1998-06-01,695.0,699.0,3.0,12.0,NaN,12.0,0.0,21929.0,64.5,35.0,w,2018-03-01,704.0,700.0,0.0,NaN,1.0,Individual,0.0,0.0,331730.0,14.0,34000.0,10.0,27644.0,4567.0,77.5,0.0,0.0,128.0,210.0,4.0,4.0,6.0,4.0,12.0,1.0,12.0,0.0,4.0,6.0,5.0,9.0,10.0,7.0,19.0,6.0,12.0,0.0,0.0,0.0,4.0,96.6,60.0,0.0,0.0,439570.0,95768.0,20300.0,88097.0,Cash
4,11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,2015-12-01,0,n,debt_consolidation,GA,10.20,0.0,1987-10-01,690.0,694.0,0.0,NaN,NaN,5.0,0.0,8822.0,68.4,6.0,w,2017-05-01,759.0,755.0,0.0,NaN,1.0,Individual,0.0,0.0,12798.0,338.0,12900.0,0.0,2560.0,844.0,91.0,0.0,0.0,338.0,54.0,32.0,32.0,0.0,36.0,NaN,NaN,NaN,0.0,2.0,3.0,2.0,2.0,2.0,4.0,4.0,3.0,5.0,0.0,0.0,0.0,0.0,100.0,100.0,0.0,0.0,16900.0,12798.0,9400.0,4000.0,Cash


## PASO FINAL: Guardar parquet


In [20]:
file_path = "../data/interim/cleaned_loans.parquet"

print("Guardando datos limpios en Parquet...")
df_completed_filtred.to_parquet(file_path, index=False)

print(f"¡Datos guardados con éxito en {file_path}!")

Guardando datos limpios en Parquet...
¡Datos guardados con éxito en ../data/interim/cleaned_loans.parquet!
